# 09 — Hybrid Scoring & Ablation

## Objective
Combine ranks from multiple detectors (IsolationForest + LOF + statistical z-score) via rank averaging and measure whether the hybrid outperforms any single model.


In [ ]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
from data_utils import load_processed
from anomaly import fit_isolation_forest, fit_lof, statistical_zscore_scores, hybrid_score
from evaluation import evaluate_anomaly_scores, summary_table

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

X_train = load_processed("X_train").drop(columns=["class"])
X_val   = load_processed("X_val")
y_val   = X_val["class"].values
X_val_f = X_val.drop(columns=["class"])

iforest = fit_isolation_forest(X_train, contamination=0.09)
lof     = fit_lof(X_train, contamination=0.09)

s_if = iforest.predict_anomaly_score(X_val_f)
s_lof = lof.predict_anomaly_score(X_val_f)
s_z  = statistical_zscore_scores(X_val_f)
s_hyb = hybrid_score({"if": s_if, "lof": s_lof, "z": s_z})

results = {
    "IsolationForest": evaluate_anomaly_scores(y_val, s_if),
    "LOF": evaluate_anomaly_scores(y_val, s_lof),
    "Z-score": evaluate_anomaly_scores(y_val, s_z),
    "Hybrid rank-avg": evaluate_anomaly_scores(y_val, s_hyb),
}
print(summary_table(results))
